<div style="text-align: center; line-height: 0; padding-top: 9px;">
<img src="https://learningjournal.github.io/pub-resources/logos/scholarnest_academy.jpg" alt="ScholarNest Academy" style="width: 1400px">
</div>

####Working with timestamp
1. [Timestamp functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html#date-and-timestamp-functions)

####1. How Spark stores the timestamp

1. Timestamp is internally stored as 12 byte integer known as INT96
2. Timestamp is made up of 7 fields
    1. Year
    2. Month
    3. Day
    4. Hour
    5. Minute
    6. Second
        * Up to 6 decimal places
        * Microsecond precision
    7. Timezone

####2. Requirement
You are given the below dataframes

In [0]:
data_list_1 = [(1, "2022-05-18T10:30:30.0000"), (2, "2022-05-19T11:30:10.0000")]
data_list_2 = [(1, "18-05-2022 10:30:30.0000"), (2, "19-05-2022 10:30:10.0000")]
data_list_3 = [(1, "2022-05-18 10:30:30.0000"), (2, "19-05-2022 10:30:10.0000")]

df_1 = (spark.createDataFrame(data_list_1).toDF("id", "string_time"))
df_2 = (spark.createDataFrame(data_list_2).toDF("id", "string_time"))
df_3 = (spark.createDataFrame(data_list_3).toDF("id", "string_time"))

2.1 covert the df_1 to timestamp

In [0]:
from pyspark.sql.functions import to_timestamp, col

df_1.select("string_time",
             to_timestamp("string_time", "yyyy-MM-dd'T'HH:mm:ss.SSSS").alias("valid_time")
             ).display()

string_time,valid_time
2022-05-18T10:30:30.0000,2022-05-18T10:30:30.000Z
2022-05-19T11:30:10.0000,2022-05-19T11:30:10.000Z


2.2 Convert the df_2 to time

In [0]:
df_2.selectExpr(
    "string_time",
    "to_timestamp(string_time, 'dd-MM-yyyy HH:mm:ss.SSSS') as valid_time"
).display()

string_time,valid_time
18-05-2022 10:30:30.0000,2022-05-18T10:30:30.000Z
19-05-2022 10:30:10.0000,2022-05-19T10:30:10.000Z


2.3 Convert the df_3 to time


In [0]:
df_3.selectExpr(
    "string_time",
    "try_to_timestamp(string_time, 'yyyy-MM-dd HH:mm:ss.SSSS') as valid_time"
).display()


string_time,valid_time
2022-05-18 10:30:30.0000,2022-05-18T10:30:30.000Z
19-05-2022 10:30:10.0000,null


####3. Timezone information

1. A timestamp without timezone information is incomplete.
2. Spark offers two data types for timestamp
    1. TIMESTAMP
    2. TIMESTAMP_NTZ
3. For TIMESTAMP, Spark assumes session timezone as the default when timezone is not specified
4. Session timezone is specified as spark.sql.session.timeZone

3.1 What is your default session timezone?


In [0]:
spark.conf.get("spark.sql.session.timeZone")

'Etc/UTC'

3.2 Change your session timezone to IST

In [0]:
spark.conf.set("spark.sql.session.timeZone", 'Etc/UTC')

####4. Working with NTZ data

4.1 Load machine-events-no-tz.csv file and show the data

In [0]:
event_ntz_schema = "component string, event_time string, reading string"

event_ntz_df = (
    spark.read.format("csv")
        .option("header", "true")
        .schema(event_ntz_schema)
        .load("/Volumes/dev/spark_db/datasets/spark_programming/data/machine-events-no-tz.csv")
)

event_ntz_df.display()

component,event_time,reading
AXT594,17-05-2022 06:14:10.359,23
AXT594,17-05-2022 06:14:25.380,25
AXT594,17-05-2022 06:14:35.346,21
AXT594,17-05-2022 06:14:45.381,22
AXT594,17-05-2022 06:14:55.356,25
AXT594,17-05-2022 06:15:05.372,23
AXT594,17-05-2022 06:15:15.355,24
AXT594,17-05-2022 06:16:25.326,24
AXT594,17-05-2022 06:17:35.345,21
AXT594,17-05-2022 06:18:45.365,22


4.2 Parse te event_time to a TIMESTAMP_NTZ value

In [0]:
from pyspark.sql.functions import to_timestamp_ntz, lit

event_valid_ntz_df = (
    event_ntz_df.withColumn("event_time_valid_ntz", to_timestamp_ntz("event_time", lit("dd-MM-yyyy HH:mm:ss.SSS")))
)

event_valid_ntz_df.display()

component,event_time,reading,event_time_valid_ntz
AXT594,17-05-2022 06:14:10.359,23,2022-05-17T06:14:10.359
AXT594,17-05-2022 06:14:25.380,25,2022-05-17T06:14:25.380
AXT594,17-05-2022 06:14:35.346,21,2022-05-17T06:14:35.346
AXT594,17-05-2022 06:14:45.381,22,2022-05-17T06:14:45.381
AXT594,17-05-2022 06:14:55.356,25,2022-05-17T06:14:55.356
AXT594,17-05-2022 06:15:05.372,23,2022-05-17T06:15:05.372
AXT594,17-05-2022 06:15:15.355,24,2022-05-17T06:15:15.355
AXT594,17-05-2022 06:16:25.326,24,2022-05-17T06:16:25.326
AXT594,17-05-2022 06:17:35.345,21,2022-05-17T06:17:35.345
AXT594,17-05-2022 06:18:45.365,22,2022-05-17T06:18:45.365


4.3 event_time_ntz field to a valid timestamp value\
Assume the event_time_ntz is IST time

In [0]:
from pyspark.sql.functions import convert_timezone, lit

stz = spark.conf.get("spark.sql.session.timeZone")

events_df = (
    event_valid_ntz_df.withColumn("event_time_tz", to_timestamp(convert_timezone(lit("IST"), lit(stz), "event_time_valid_ntz")))
)

events_df.display()

component,event_time,reading,event_time_valid_ntz,event_time_tz
AXT594,17-05-2022 06:14:10.359,23,2022-05-17T06:14:10.359,2022-05-17T00:44:10.359Z
AXT594,17-05-2022 06:14:25.380,25,2022-05-17T06:14:25.380,2022-05-17T00:44:25.380Z
AXT594,17-05-2022 06:14:35.346,21,2022-05-17T06:14:35.346,2022-05-17T00:44:35.346Z
AXT594,17-05-2022 06:14:45.381,22,2022-05-17T06:14:45.381,2022-05-17T00:44:45.381Z
AXT594,17-05-2022 06:14:55.356,25,2022-05-17T06:14:55.356,2022-05-17T00:44:55.356Z
AXT594,17-05-2022 06:15:05.372,23,2022-05-17T06:15:05.372,2022-05-17T00:45:05.372Z
AXT594,17-05-2022 06:15:15.355,24,2022-05-17T06:15:15.355,2022-05-17T00:45:15.355Z
AXT594,17-05-2022 06:16:25.326,24,2022-05-17T06:16:25.326,2022-05-17T00:46:25.326Z
AXT594,17-05-2022 06:17:35.345,21,2022-05-17T06:17:35.345,2022-05-17T00:47:35.345Z
AXT594,17-05-2022 06:18:45.365,22,2022-05-17T06:18:45.365,2022-05-17T00:48:45.365Z


####5. Working with TZ data

5.1 Load machine-events-with-tz.csv file and show the data

In [0]:
event_tz_schema = "component string, event_time_tz_str string, reading string"

events_tz_df = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(event_tz_schema)
    .load("/Volumes/dev/spark_db/datasets/spark_programming/data/machine-events-with-tz.csv")
)

display(events_tz_df)

component,event_time_tz_str,reading
AXT594,17-05-2022 06:14:10.359+0000,23
AXT595,17-05-2022 06:14:25.380+0530,22
AXT596,17-05-2022 06:14:35.346+0100,24
AXT594,17-05-2022 06:14:45.381+0000,21
AXT595,17-05-2022 06:14:55.356+0530,23
AXT596,17-05-2022 06:15:05.372+0100,22
AXT594,17-05-2022 06:15:15.355+0000,21
AXT595,17-05-2022 06:16:25.326+0530,25
AXT596,17-05-2022 06:17:35.345+0100,22
AXT594,17-05-2022 06:18:45.365+0000,21


5.2 Parse the event_time field to a valid timestamp value\
Timezone information is provided in the data file

In [0]:
from pyspark.sql.functions import to_timestamp
event_data_df = (
    events_tz_df.withColumn("event_time_tz", to_timestamp("event_time_tz_str", "dd-MM-yyyy HH:mm:ss.SSSZ"))
)
event_data_df.display()

component,event_time_tz_str,reading,event_time_tz
AXT594,17-05-2022 06:14:10.359+0000,23,2022-05-17T06:14:10.359Z
AXT595,17-05-2022 06:14:25.380+0530,22,2022-05-17T00:44:25.380Z
AXT596,17-05-2022 06:14:35.346+0100,24,2022-05-17T05:14:35.346Z
AXT594,17-05-2022 06:14:45.381+0000,21,2022-05-17T06:14:45.381Z
AXT595,17-05-2022 06:14:55.356+0530,23,2022-05-17T00:44:55.356Z
AXT596,17-05-2022 06:15:05.372+0100,22,2022-05-17T05:15:05.372Z
AXT594,17-05-2022 06:15:15.355+0000,21,2022-05-17T06:15:15.355Z
AXT595,17-05-2022 06:16:25.326+0530,25,2022-05-17T00:46:25.326Z
AXT596,17-05-2022 06:17:35.345+0100,22,2022-05-17T05:17:35.345Z
AXT594,17-05-2022 06:18:45.365+0000,21,2022-05-17T06:18:45.365Z


&copy; 2021-2026 <a href="https://www.scholarnest.com/">ScholarNest</a>. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the <a href="https://www.apache.org/">Apache Software Foundation.</a><br/>
Databricks, Databricks Cloud and the Databricks logo are trademarks of the <a href="https://www.databricks.com/">Databricks Inc.</a><br/>
<a href="https://www.scholarnest.com/pages/privacy">Privacy Policy</a> | <a href="https://www.scholarnest.com/pages/terms">Terms of Use</a> | <a href="https://www.scholarnest.com/pages/contact">Contact Us</a>